# 📚 Book Recommendation Engine using K-Nearest Neighbors

## Project Overview
This notebook builds a **collaborative filtering book recommendation system** using the K-Nearest Neighbors algorithm. Given a book title, the engine returns the 5 most similar books based on user rating patterns.

### Dataset: Book-Crossings
- **1.1 million ratings** (scale 1–10)
- **270,000 books**
- **90,000 users**

### Algorithm: K-Nearest Neighbors (KNN)
KNN measures **cosine distance** between books in a user-rating vector space. Books with similar rating patterns across users are considered similar.

---

## Step 1: Import Libraries

In [ ]:
# Core data manipulation
import numpy as np
import pandas as pd

# Sparse matrix (memory-efficient for large rating matrices)
from scipy.sparse import csr_matrix

# KNN model from scikit-learn
from sklearn.neighbors import NearestNeighbors

# Visualization (optional)
import matplotlib.pyplot as plt
import seaborn as sns

print('✅ Libraries imported successfully!')

## Step 2: Load the Dataset

The Book-Crossings dataset has three files:
- `BX-Books.csv` — book metadata (title, author, year, publisher)
- `BX-Users.csv` — user demographics
- `BX-Book-Ratings.csv` — user-book ratings (the key file for recommendations)

In [ ]:
# Load datasets from the FreeCodeCamp GitHub mirror
# These URLs are pre-configured for the FCC project environment

books_url   = 'https://cdn.freecodecamp.org/project-data/books/book_data.csv'
ratings_url = 'https://cdn.freecodecamp.org/project-data/books/user-book.csv'

# Load books: ISBN, Title, Author, Year, Publisher
books = pd.read_csv(
    books_url,
    encoding='Isbn,Title,Author,Year,Publisher'.split(','),  # explicit column names for safety
    sep=',',
    header=0,
    low_memory=False
)

# Load ratings: User-ID, ISBN, Book-Rating
ratings = pd.read_csv(
    ratings_url,
    encoding='User-ID,ISBN,Book-Rating'.split(','),
    sep=',',
    header=0,
    low_memory=False
)

print(f'Books shape:   {books.shape}')
print(f'Ratings shape: {ratings.shape}')

In [ ]:
# ─── ALTERNATIVE LOAD (use this block if above fails) ───────────────────────
# If running in the official FCC Colab, the data is already loaded as:
#   df_books and df_ratings
# Uncomment and rename if needed:
#
# books   = df_books.copy()
# ratings = df_ratings.copy()
# ────────────────────────────────────────────────────────────────────────────

# Preview the data
print('=== Books Sample ===')
print(books.head(3))
print('\n=== Ratings Sample ===')
print(ratings.head(3))

## Step 3: Explore the Data (EDA)

In [ ]:
# Basic statistics on ratings
print('=== Rating Distribution ===')
print(ratings.iloc[:, 2].value_counts().sort_index())  # column 2 = Book-Rating

# How many unique users and books in the ratings file?
print(f'\nUnique users in ratings: {ratings.iloc[:, 0].nunique():,}')
print(f'Unique books in ratings: {ratings.iloc[:, 1].nunique():,}')

In [ ]:
# (Optional) Visualize the distribution of ratings per user and per book
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Ratings per user
user_rating_counts = ratings.iloc[:, 0].value_counts()
axes[0].hist(user_rating_counts, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Ratings per User (all users)')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

# Ratings per book
book_rating_counts = ratings.iloc[:, 1].value_counts()
axes[1].hist(book_rating_counts, bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Ratings per Book (all books)')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Count')
axes[1].set_yscale('log')

plt.suptitle('Long-Tail Distribution in Book-Crossings Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
# 💡 Most books and users have very few ratings — this is the classic "long tail"

## Step 4: Data Cleaning & Filtering

To ensure **statistical significance**, we:
1. Remove users with **fewer than 200 ratings** (too sparse to draw conclusions)
2. Remove books with **fewer than 100 ratings** (not enough signal)

In [ ]:
# ── Rename columns for easier reference ──────────────────────────────────────
# Standardise column names regardless of original header names
ratings.columns = ['user_id', 'isbn', 'rating']
books.columns   = ['isbn', 'title', 'author', 'year', 'publisher']

print('Columns standardised ✅')
print('Ratings:', ratings.columns.tolist())
print('Books:  ', books.columns.tolist())

In [ ]:
# ── Filter 1: Keep only users with ≥ 200 ratings ─────────────────────────────
user_counts = ratings['user_id'].value_counts()
active_users = user_counts[user_counts >= 200].index          # users who rated 200+ books

ratings_filtered = ratings[ratings['user_id'].isin(active_users)]
print(f'Users before filter: {ratings["user_id"].nunique():,}')
print(f'Users after  filter: {ratings_filtered["user_id"].nunique():,}')

In [ ]:
# ── Filter 2: Keep only books with ≥ 100 ratings ─────────────────────────────
book_counts = ratings_filtered['isbn'].value_counts()
popular_books = book_counts[book_counts >= 100].index         # books rated 100+ times

ratings_filtered = ratings_filtered[ratings_filtered['isbn'].isin(popular_books)]
print(f'Ratings after both filters: {len(ratings_filtered):,}')
print(f'Unique books remaining:     {ratings_filtered["isbn"].nunique():,}')

In [ ]:
# ── Merge ratings with book titles ───────────────────────────────────────────
# We only need ISBN → title mapping from books
books_meta = books[['isbn', 'title']].drop_duplicates('isbn')

# Inner join: keeps only books present in both dataframes
df = ratings_filtered.merge(books_meta, on='isbn', how='inner')

print(f'Merged dataframe shape: {df.shape}')
print(df.head(3))

## Step 5: Build the Book-User Matrix

KNN works on a **feature matrix**. Each book is a row; each user is a column.
The value is the rating given by that user for that book (0 if not rated).

We use a **sparse matrix** (`csr_matrix`) to handle the large number of zeros efficiently.

In [ ]:
# ── Deduplicate: one rating per (user, book) pair ────────────────────────────
# If a user rated a book multiple times, keep the mean
df_pivot = df.groupby(['title', 'user_id'])['rating'].mean().reset_index()

# ── Pivot to wide format: rows = books, columns = users ──────────────────────
book_user_matrix = df_pivot.pivot(
    index='title',
    columns='user_id',
    values='rating'
).fillna(0)  # NaN means the user didn't rate → treat as 0

print(f'Book-User Matrix shape: {book_user_matrix.shape}')
print(f'  Rows    = books: {book_user_matrix.shape[0]:,}')
print(f'  Columns = users: {book_user_matrix.shape[1]:,}')

In [ ]:
# ── Convert to sparse matrix for memory efficiency ───────────────────────────
# Most cells are 0 (users haven't rated most books) → sparse is ~10x more efficient
mat = csr_matrix(book_user_matrix.values)

# Keep the book titles list so we can look up by index
book_titles = book_user_matrix.index.tolist()

print(f'Sparse matrix created: {mat.shape} | nnz (non-zero) = {mat.nnz:,}')

## Step 6: Train the KNN Model

We use `NearestNeighbors` with:
- **`metric='cosine'`** — measures the angle between rating vectors (better than Euclidean for sparse data)
- **`algorithm='brute'`** — exact search (suitable for our matrix size)

In [ ]:
# ── Instantiate and fit the KNN model ────────────────────────────────────────
# n_neighbors=6: returns the book itself + 5 recommendations
model_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=6,   # 1 (itself) + 5 recommendations
    n_jobs=-1        # use all CPU cores
)

# Fit on the sparse book-user matrix
model_knn.fit(mat)

print('✅ KNN model trained!')

## Step 7: Build the `get_recommends` Function

The function:
1. Finds the book's row index in the matrix
2. Queries KNN for the 6 nearest neighbours (including itself)
3. Returns the book title + a list of `[recommended_title, distance]` pairs

In [ ]:
def get_recommends(book_title: str):
    """
    Returns 5 book recommendations similar to the given title.

    Parameters
    ----------
    book_title : str
        Exact book title as it appears in the dataset.

    Returns
    -------
    list : [book_title, [[rec_title, distance], ...]]
        - First element  : the query book title
        - Second element : list of 5 [recommended_title, cosine_distance] pairs,
                           sorted from MOST similar (highest distance) to LEAST similar
    """
    # ── Step 1: Look up the book's index in our matrix ────────────────────────
    if book_title not in book_titles:
        raise ValueError(f'Book "{book_title}" not found in the dataset. '
                         f'Check the exact title spelling.')

    book_idx = book_titles.index(book_title)   # row index in book_user_matrix

    # ── Step 2: Get the book's rating vector (1 row from the sparse matrix) ───
    book_vector = mat[book_idx]                # shape: (1, n_users)

    # ── Step 3: Query KNN — returns distances & indices of 6 nearest books ────
    distances, indices = model_knn.kneighbors(
        book_vector,
        n_neighbors=6   # includes the book itself at index 0
    )

    # ── Step 4: Flatten results (kneighbors returns 2D arrays) ───────────────
    distances = distances.flatten()  # [0.0, 0.52, 0.54, ...]  (0.0 = book itself)
    indices   = indices.flatten()    # [book_idx, idx1, idx2, ...]

    # ── Step 5: Build the recommendation list ────────────────────────────────
    # Skip index 0 (the book itself, distance = 0), keep 1–5
    # Reverse so highest-distance (most similar by cosine) comes first
    recommended = []
    for i in range(1, 6):                          # indices 1–5 → 5 neighbours
        rec_title    = book_titles[indices[i]]
        rec_distance = float(distances[i])
        recommended.append([rec_title, rec_distance])

    # Sort from FARTHEST to CLOSEST (highest cosine distance first)
    # This matches the expected FCC output format
    recommended = sorted(recommended, key=lambda x: x[1], reverse=True)

    return [book_title, recommended]


print('✅ get_recommends() function defined!')

## Step 8: Test the Function

In [ ]:
# ── Quick test ────────────────────────────────────────────────────────────────
test_title = "The Queen of the Damned (Vampire Chronicles (Paperback))"
result = get_recommends(test_title)

print(f'Query book: {result[0]}')
print('\nTop 5 Recommendations:')
for rank, (title, dist) in enumerate(result[1], start=1):
    print(f'  {rank}. {title:60s}  [distance: {dist:.4f}]')

In [ ]:
# ── Try another book ──────────────────────────────────────────────────────────
result2 = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(f'Query: {result2[0]}')
print('Recommendations:')
for title, dist in result2[1]:
    print(f'  • {title} ({dist:.4f})')

## Step 9: Run the Official FreeCodeCamp Test

In [ ]:
def test_book_recommendation():
    """
    FreeCodeCamp test suite for the Book Recommendation Engine.
    Tests exact book titles and approximate distance thresholds.
    """
    test_pass = True
    recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")

    # ── Test 1: Check query title is returned as first element ────────────────
    if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
        print('FAIL: query title not returned correctly')
        test_pass = False

    # ── Test 2: Check 5 recommendations are returned ──────────────────────────
    if len(recommends[1]) != 5:
        print(f'FAIL: expected 5 recommendations, got {len(recommends[1])}')
        test_pass = False

    # ── Test 3: Check known expected books appear in results ──────────────────
    expected_books   = ['I Know This Much Is True']
    expected_dists   = [0.7677856]

    returned_titles = [r[0] for r in recommends[1]]
    returned_dists  = [r[1] for r in recommends[1]]

    for exp_book, exp_dist in zip(expected_books, expected_dists):
        if exp_book not in returned_titles:
            print(f'FAIL: "{exp_book}" not found in recommendations')
            test_pass = False
        else:
            idx  = returned_titles.index(exp_book)
            dist = returned_dists[idx]
            if abs(dist - exp_dist) > 0.05:  # allow small tolerance
                print(f'FAIL: distance for "{exp_book}" expected ~{exp_dist}, got {dist:.4f}')
                test_pass = False

    if test_pass:
        print('✅ You passed the challenge!')
    else:
        print('❌ Tests failed. Review the output above.')


test_book_recommendation()

---
## 📝 Summary

| Step | Action |
|------|--------|
| 1 | Loaded Book-Crossings dataset (ratings + metadata) |
| 2 | Filtered: ≥200 ratings/user, ≥100 ratings/book |
| 3 | Built a sparse Book × User rating matrix |
| 4 | Trained KNN (cosine distance, brute-force) |
| 5 | Created `get_recommends()` to query similar books |

### Key Takeaways
- **Collaborative filtering** leverages the wisdom of crowds — similar tastes → similar recommendations
- **Cosine distance** outperforms Euclidean for sparse rating matrices
- **Filtering low-activity users/books** is essential to avoid noise and improve recommendation quality